In [1]:
!pip install transformers

In [2]:
import torch

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [4]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [8]:
sequence = "I've been waiting for a HuggingFace course my whole life."

In [9]:
tokens = tokenizer.tokenize(sequence)

In [10]:
print(tokens)

['i', "'", 've', 'been', 'waiting', 'for', 'a', 'hugging', '##face', 'course', 'my', 'whole', 'life', '.']


In [12]:
ids = tokenizer.convert_tokens_to_ids(tokens)

In [14]:
print(ids)

[1045, 1005, 2310, 2042, 3403, 2005, 1037, 17662, 12172, 2607, 2026, 2878, 2166, 1012]


In [15]:
input_ids = torch.tensor(ids)

In [18]:
print(input_ids)

tensor([ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
         2026,  2878,  2166,  1012])


In [17]:
# model(input_ids)  -> this line will fail

In [19]:
tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print(tokenized_inputs["input_ids"])

tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])


In [20]:
input_ids = torch.tensor([ids])

In [21]:
print("Input IDs:", input_ids)

Input IDs: tensor([[ 1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,  2607,
          2026,  2878,  2166,  1012]])


In [22]:
output = model(input_ids)

In [23]:
print("Logits:", output.logits)

Logits: tensor([[-2.7276,  2.8789]], grad_fn=<AddmmBackward0>)


### Batching is the act of sending multiple sentences through the model, all at once. If you only have one sentence, you can just build a batch with a single sequence:

In [24]:
batched_ids = [ids, ids]

In [27]:
batched_ids = [
    [200, 200, 200],
    [200, 200]
]

In [28]:
padding_id = 100

batched_ids = [
    [200, 200, 200],
    [200, 200, padding_id],
]

In [29]:
sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

In [30]:
print(model(torch.tensor(sequence1_ids)).logits)

tensor([[ 1.5694, -1.3895]], grad_fn=<AddmmBackward0>)


In [31]:
print(model(torch.tensor(sequence2_ids)).logits)

tensor([[ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


In [32]:
print(model(torch.tensor(batched_ids)).logits)

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 1.5694, -1.3895],
        [ 1.3373, -1.2163]], grad_fn=<AddmmBackward0>)


## Attention Masks

In [33]:
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

In [34]:
attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

In [35]:
outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

tensor([[ 1.5694, -1.3895],
        [ 0.5803, -0.4125]], grad_fn=<AddmmBackward0>)


In [36]:
tokens = tokenizer.tokenize("I am Arshvir")
print(tokens)               # Shows subword tokens
print(len(tokens))

['i', 'am', 'ars', '##h', '##vir']
5
